In [8]:
# ============================================================
# DECODE LABS - SYSTEMATIC TRADING ENGINE
# Project 1: Technical Analysis & Price Action
# Author: Stephanie
# ============================================================

import yfinance as yf
import pandas as pd

# ------------------------------------------------------------
# STEP 1: FETCH RAW PRICE DATA
# We need at least 200 days for our 200-day EMA to work
# We'll use 2 years to have plenty of data
# ------------------------------------------------------------

def fetch_data(ticker):
    """
    Downloads historical OHLC price data for a given stock.

    Parameters:
        ticker (str): The stock ticker symbol e.g. 'AAPL'

    Returns:
        df (DataFrame): A dataframe containing daily OHLC and Volume data
    """
    df = yf.download(ticker, period='2y')
    return df

# Test it
df = fetch_data('AAPL')
print(df.head())

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open     Volume
Ticker            AAPL        AAPL        AAPL        AAPL       AAPL
Date                                                                 
2024-06-17  214.809204  217.069622  210.893130  211.537542   93728300
2024-06-18  212.449646  216.752385  211.170731  215.721308   79943300
2024-06-20  207.879227  212.400077  207.056368  212.092727   86172500
2024-06-21  205.708069  210.070275  205.331327  208.583157  241805100
2024-06-24  206.352478  210.873314  204.815787  205.936087   80727000


In [10]:
def calculate_rwb(df):
    """
    Calculates the Wick-to-Body Ratio for each candle.

    R_wb = (Upper Wick + Lower Wick) / Body
    """
    open_price = df['Open'].squeeze()
    high_price = df['High'].squeeze()
    low_price = df['Low'].squeeze()
    close_price = df['Close'].squeeze()

    upper_wick = high_price - pd.concat([open_price, close_price], axis=1).max(axis=1)
    lower_wick = pd.concat([open_price, close_price], axis=1).min(axis=1) - low_price
    body = abs(open_price - close_price)

    df['Upper_Wick'] = upper_wick
    df['Lower_Wick'] = lower_wick
    df['Body'] = body
    df['R_wb'] = (upper_wick + lower_wick) / body.replace(0, 0.0001)

    return df

df = calculate_rwb(df)
print(df[['Open', 'High', 'Low', 'Close', 'R_wb']].head())

Price             Open        High         Low       Close       R_wb
Ticker            AAPL        AAPL        AAPL        AAPL           
Date                                                                 
2024-06-17  211.537542  217.069622  210.893130  214.809204   0.887876
2024-06-18  215.721308  216.752385  211.170731  212.449646   0.706061
2024-06-20  212.092727  212.400077  207.056368  207.879227   0.268235
2024-06-21  208.583157  210.070275  205.331327  205.708069   0.648279
2024-06-24  205.936087  210.873314  204.815787  206.352478  13.547684


In [11]:
# ------------------------------------------------------------
# STEP 3: CALCULATE 50 AND 200 DAY EMA
# 50 EMA = short term trend
# 200 EMA = long term trend
# Golden Cross = 50 crosses above 200 = bullish
# Death Cross = 50 crosses below 200 = bearish
# ------------------------------------------------------------

def calculate_ema(df):
    """
    Calculates 50-day and 200-day Exponential Moving Averages.
    Detects Golden Cross and Death Cross signals.
    """
    close = df['Close'].squeeze()

    df['EMA_50'] = close.ewm(span=50, adjust=False).mean()
    df['EMA_200'] = close.ewm(span=200, adjust=False).mean()

    return df

df = calculate_ema(df)
print(df[['Close', 'EMA_50', 'EMA_200']].head(10))

Price            Close      EMA_50     EMA_200
Ticker            AAPL                        
Date                                          
2024-06-17  214.809204  214.809204  214.809204
2024-06-18  212.449646  214.716672  214.785726
2024-06-20  207.879227  214.448537  214.717005
2024-06-21  205.708069  214.105774  214.627363
2024-06-24  206.352478  213.801723  214.545026
2024-06-25  207.274506  213.545754  214.472683
2024-06-26  211.418610  213.462336  214.442294
2024-06-27  212.261322  213.415238  214.420593
2024-06-28  208.811188  213.234687  214.364778
2024-07-01  214.888550  213.299544  214.369989


In [12]:
# ------------------------------------------------------------
# STEP 4: DETECT GOLDEN CROSS AND DEATH CROSS
# Golden Cross = 50 EMA crosses ABOVE 200 EMA = Bullish
# Death Cross = 50 EMA crosses BELOW 200 EMA = Bearish
# ------------------------------------------------------------

def detect_crossovers(df):
    """
    Detects when the 50 EMA crosses the 200 EMA.
    Golden Cross = bullish signal
    Death Cross = bearish signal
    """
    df['Golden_Cross'] = (df['EMA_50'] > df['EMA_200']) & (df['EMA_50'].shift(1) <= df['EMA_200'].shift(1))
    df['Death_Cross'] = (df['EMA_50'] < df['EMA_200']) & (df['EMA_50'].shift(1) >= df['EMA_200'].shift(1))

    return df

df = detect_crossovers(df)

# Show only the days where a crossover actually happened
crossovers = df[df['Golden_Cross'] | df['Death_Cross']]
print(crossovers[['Close', 'EMA_50', 'EMA_200', 'Golden_Cross', 'Death_Cross']])

Price            Close      EMA_50     EMA_200 Golden_Cross Death_Cross
Ticker            AAPL                                                 
Date                                                                   
2024-06-18  212.449646  214.716672  214.785726        False        True
2024-07-09  226.716080  215.091259  214.793127         True       False
2025-03-31  220.962494  227.583871  227.692434        False        True
2025-08-26  228.663086  215.998182  215.838625         True       False


In [13]:
# ------------------------------------------------------------
# STEP 5: CALCULATE RSI
# Measures momentum - speed and strength of price movement
# RSI above 50 = buyers in control = Gate C passes
# RSI below 50 = sellers in control = Gate C fails
# Standard lookback period = 14 days
# ------------------------------------------------------------

def calculate_rsi(df, period=14):
    """
    Calculates the Relative Strength Index (RSI).

    Parameters:
        df: dataframe with price data
        period: lookback period (default 14 days)

    Returns:
        df with RSI column added
    """
    close = df['Close'].squeeze()

    # Calculate daily price changes
    delta = close.diff()

    # Separate gains and losses
    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)

    # Calculate average gains and losses over 14 days
    avg_gain = gains.ewm(span=period, adjust=False).mean()
    avg_loss = losses.ewm(span=period, adjust=False).mean()

    # Calculate RS and RSI
    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))

    return df

df = calculate_rsi(df)
print(df[['Close', 'RSI']].tail(10))

Price            Close        RSI
Ticker            AAPL           
Date                             
2026-06-02  315.200012  73.510069
2026-06-03  310.260010  60.241162
2026-06-04  311.230011  61.803258
2026-06-05  307.339996  52.295740
2026-06-08  301.540009  41.351795
2026-06-09  290.549988  28.370994
2026-06-10  291.579987  30.722706
2026-06-11  295.630005  39.704234
2026-06-12  291.130005  34.045465
2026-06-15         NaN  34.045465


In [16]:
# ------------------------------------------------------------
# STEP 6: THE DECISION ENGINE - 3 GATE SYSTEM
# Gate A: Is price near a support or resistance level?
# Gate B: Is 50 EMA above 200 EMA? (Bullish regime)
# Gate C: Is RSI above 50? (Momentum confirmed)
# All three gates must pass → EXECUTE
# Any gate fails → IDLE
# ------------------------------------------------------------

def decision_engine(df):
    """
    Runs all three gates on the latest data point.
    Returns a trading signal: EXECUTE or IDLE
    """
    # Get the most recent row
    latest = df.iloc[-1]

    # Gate A: Is price within 2% of recent high or low?
    recent_high = df['Close'].squeeze().tail(20).max()
    recent_low = df['Close'].squeeze().tail(20).min()
    price = latest['Close'].squeeze()

    near_resistance = price >= recent_high * 0.98
    near_support = price <= recent_low * 1.02
    gate_a = near_resistance or near_support

    # Gate B: Is 50 EMA above 200 EMA?
    gate_b = latest['EMA_50'] > latest['EMA_200']

    # Gate C: Is RSI above 50?
    gate_c = latest['RSI'] > 50

    # Print each gate status
    print(f"Gate A - Price at structure: {gate_a}")
    print(f"Gate B - Bullish EMA regime: {gate_b}")
    print(f"Gate C - RSI momentum > 50: {gate_c}")
    print("---")

    # Final decision
    if gate_a and gate_b and gate_c:
        print("SIGNAL: EXECUTE ")
    else:
        print("SIGNAL: IDLE ")

decision_engine(df)

Gate A - Price at structure: False
Gate B - Bullish EMA regime: Ticker
    True
Name: 2026-06-15 00:00:00, dtype: bool
Gate C - RSI momentum > 50: Ticker
    False
Name: 2026-06-15 00:00:00, dtype: bool
---
SIGNAL: IDLE 
